In [2]:
import os
import ast
import re
import pandas as pd

CV_runs = pd.read_csv("llm_pipeline/outputs_export/cv_runs.csv")
df_openai = CV_runs[CV_runs["provider"] == "openai"]
df_gemini = CV_runs[CV_runs["provider"] == "google-gemini"]

OUTPUT_OPENAI = "openai_cvs_tex_harvard_ordered"
OUTPUT_GEMINI = "gemini_cvs_tex_harvard_ordered"

FileNotFoundError: [Errno 2] No such file or directory: 'llm_pipeline/outputs_export/cv_runs.csv'

In [11]:
# Helpers for LaTeX generation

def latex_escape(text):
    if text is None:
        return ""
    text = str(text)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def slugify(s):
    s = str(s)
    s = s.lower()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^a-z0-9_]+", "", s)
    return s


def parse_response_json(raw):
    # raw ist ein Python-Dict als String, kein echtes JSON
    return ast.literal_eval(raw)


def section_title(num, title):
    # Nummerierung fest reinschreiben
    return rf"\section{{{num}. {latex_escape(title)}}}"


def format_personal_data_section(num, perso):
    if not perso:
        return ""

    lines = [section_title(num, "Persönliche Daten")]

    # Alles, was in 01_persoenliche_daten drin ist, aufführen:
    # Name, Adresse, Telefon, E-Mail, Geburtsdatum, Alter, Einsatzort, etc.
    lines.append(r"\begin{itemize}[leftmargin=1.2em,itemsep=0.2em]")
    for k, v in perso.items():
        if v is None:
            continue
        key = latex_escape(str(k))
        val = latex_escape(str(v))
        lines.append(r"\item " + rf"\textbf{{{key}}}: {val}")
    lines.append(r"\end{itemize}")

    return "\n".join(lines)


def format_text_section(num, title, content):
    if not content:
        return ""
    lines = [section_title(num, title)]
    lines.append(latex_escape(content))
    return "\n".join(lines)


def format_list_section(num, title, content):
    if not content:
        return ""
    lines = [section_title(num, title)]

    if isinstance(content, str):
        lines.append(latex_escape(content))
        return "\n".join(lines)

    if isinstance(content, list):
        lines.append(r"\begin{itemize}[leftmargin=1.2em,itemsep=0.2em]")
        for c in content:
            lines.append(r"\item " + latex_escape(c))
            lines.append(r"\end{itemize}")
            return "\n".join(lines)

    if isinstance(content, dict):
        lines.append(r"\begin{itemize}[leftmargin=1.2em,itemsep=0.2em]")
        for k, v in content.items():
            if v is None:
                continue
            lines.append(r"\item " + latex_escape(f"{k}: {v}"))
        lines.append(r"\end{itemize}")
        return "\n".join(lines)



def format_experience_section(num, exp):
    if not exp:
        return ""

    lines = [section_title(num, "Berufserfahrung")]

    if isinstance(exp, str):
        lines.append(latex_escape(exp))
        return "\n".join(lines)

    for job in exp:
        if not isinstance(job, dict):
            continue

        company = latex_escape(
            job.get("Unternehmen")
            or job.get("unternehmen")
            or job.get("company")
            or ""
        )
        position = latex_escape(
            job.get("Position")
            or job.get("position")
            or job.get("job_title")
            or job.get("stelle")
            or ""
        )
        duration = latex_escape(
            job.get("Zeitraum")
            or job.get("zeitraum")
            or job.get("Dauer")
            or job.get("duration")
            or ""
        )
        ort = latex_escape(job.get("Ort") or job.get("ort") or "")

        header = ""
        if position:
            header = r"\textbf{" + position + "}"
        if company:
            if header:
                header += r" \hfill " + company
            else:
                header = company
        if duration:
            header += r" \hfill " + duration
        lines.append(header + r"\\")

        if ort:
            lines.append(ort + r"\\")

        beschreibung = (
            job.get("Beschreibung")
            or job.get("beschreibung")
            or job.get("Aufgabenbeschreibung")
            or job.get("aufgabenbeschreibung")
        )
        if beschreibung:
            lines.append(latex_escape(beschreibung) + r"\\")

        aufgaben = job.get("Aufgaben") or job.get("aufgaben")
        if isinstance(aufgaben, list) and aufgaben:
            lines.append(r"\begin{itemize}[leftmargin=1.2em,itemsep=0.2em]")
            for a in aufgaben:
                lines.append(r"\item " + latex_escape(a))
            lines.append(r"\end{itemize}")

        lines.append(r"\vspace{0.4em}")

    return "\n".join(lines)


def format_education_section(num, edu):
    if not edu:
        return ""

    lines = [section_title(num, "Ausbildung")]

    if isinstance(edu, str):
        lines.append(latex_escape(edu))
        return "\n".join(lines)

    for e in edu:
        if not isinstance(e, dict):
            continue

        abschluss = latex_escape(
            e.get("Abschluss") or e.get("abschluss") or e.get("degree") or ""
        )
        field = latex_escape(
            e.get("Studienrichtung")
            or e.get("field")
            or e.get("studiengang")
            or ""
        )
        inst = latex_escape(
            e.get("Institution")
            or e.get("institution")
            or e.get("Universität")
            or e.get("Universitaet")
            or ""
        )
        year = latex_escape(
            e.get("Jahr") or e.get("year") or e.get("Abschlussjahr") or ""
        )

        line = ""
        if abschluss:
            line = r"\textbf{" + abschluss + "}"
        if field:
            if line:
                line += r", " + field
            else:
                line = field
        if inst:
            if line:
                line += r", " + inst
            else:
                line = inst
        if year:
            line += r" \hfill " + year

        if line:
            lines.append(line + r"\\")

    return "\n".join(lines)


In [12]:
# build LaTeX CV
def build_cv_latex(meta, data):
    perso = data.get("01_persoenliche_daten", {}) or {}
    profil = data.get("02_profil", "")
    faehigkeiten = data.get("03_faehigkeiten", "")
    berufserfahrung = data.get("04_berufserfahrung", "")
    ausbildung = data.get("05_ausbildung", "")
    skills = data.get("06_skills", "")
    sprachen = data.get("07_sprachen", "")
    interessen = data.get("08_interessen", "")
    ziel = data.get("09_angestrebte_position", "")
    cover = data.get("10_cover_letter_snippet", "")

    # Name extrahieren
    name = (
        perso.get("Name")
        or perso.get("name")
        or f"{meta['first_name']} {meta['last_name']}"
    )

    header = r"""\documentclass[9pt,a4paper]{article}
\usepackage[margin=1.8cm]{geometry}
\usepackage[ngerman]{babel}
\usepackage[T1]{fontenc}
\usepackage{lmodern}
\usepackage{microtype}
\usepackage{enumitem}
\usepackage[hidelinks]{hyperref}
\usepackage{titlesec}

\setlength{\parskip}{3pt}
\setlength{\parindent}{0pt}

\titleformat{\section}{\large\bfseries\scshape}{}{0pt}{}[\titlerule]

\begin{document}
\small
"""

   # Name als Überschrift
    name_block = r"{\huge\bfseries " + latex_escape(name) + r"}" + "\n\n"

    # 1. Persönliche Daten
    sec1 = format_personal_data_section(1, perso)

    # 2. Profil
    sec2 = format_text_section(2, "Profil", profil) if profil else ""

    # 3. Fähigkeiten
    sec3 = format_list_section(3, "Fähigkeiten", faehigkeiten)

    # 4. Berufserfahrung
    sec4 = format_experience_section(4, berufserfahrung)

    # 5. Ausbildung
    sec5 = format_education_section(5, ausbildung)

    # 6. Skills
    sec6 = format_list_section(6, "Skills", skills)

    # 7. Sprachen
    sec7 = format_list_section(7, "Sprachen", sprachen)

    # 8. Interessen
    sec8 = format_list_section(8, "Interessen", interessen)

    # 9. Angestrebte Position
    sec9 = format_text_section(9, "Angestrebte Position", ziel) if ziel else ""

    # 10. Cover Letter Snippet
    sec10 = format_text_section(10, "Cover Letter Snippet", cover) if cover else ""

    sections = [sec1, sec2, sec3, sec4, sec5, sec6, sec7, sec8, sec9, sec10]
    body = "\n\n".join(s for s in sections if s)

    footer = "\n\\end{document}\n"

    return header + name_block + body + footer



In [14]:
# Main: 

def main(df, output_dir):
    # sicherstellen, dass der Output-Ordner existiert
    os.makedirs(output_dir, exist_ok=True)

    
    for idx, row in df.iterrows():
        try:
            data = parse_response_json(row["response_json"])
        except (SyntaxError, ValueError):
            continue

        meta = {
            "profile_id": row.get("profile_id"),
            "first_name": row.get("first_name", ""),
            "last_name": row.get("last_name", ""),
        }

        tex = build_cv_latex(meta, data)

        pid = meta["profile_id"]
        fname = slugify(meta["first_name"])
        lname = slugify(meta["last_name"])
        filename = f"cv_{pid}_{fname}_{lname}.tex"

        out_path = os.path.join(output_dir, filename)
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(tex)

    print("Finished writing LaTeX CVs to", output_dir)


if __name__ == "__main__":
    # erster Lauf
    # main(df_openai, OUTPUT_OPENAI)
    # zweiter Lauf  
    main(df_gemini, OUTPUT_GEMINI)

Finished writing LaTeX CVs to gemini_cvs_tex_harvard_ordered
